In [1]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# 変数およびパラメータを定義する
c0, c1, k0, k1 = sp.symbols("c0, c1, k0, k1")
α, β, δ = sp.symbols("α, β, δ")

# パラメータの値を定義する
params = {α: 0.3,
          β: 0.99,
          δ: 0.25
         }

# モデルの関数を定義する
f1 = c1/c0-β*(α*k1**(α-1)-δ+1)
f2 = k1-k0**α-(1-δ)*k0+c0

# 関数の行列を作る
F = sp.Matrix([f1, f2]).subs({ #ついでに定常状態を仮定した
    c1: c0, k1: k0
})
F_param = F.subs(params) # パラメータに実数を代入する

# jacobian行列を作成する関数
def Jacobian(F, N, X): # F: 元の関数, N: 変数の数, X: 変数のリスト
    J = []
    for j in range(N):
        a = [F[j].diff(i) for i in X] # 第j行の関数を各変数で微分する
        J.append(a)
    
    J = sp.Matrix(J) # 返す結果をsympyの形にしておく
    
    return J

N = 2
X = [c0, k0]
J_param = Jacobian(F_param, N, X)


# 定常状態における変数の値を求める

x_init = np.ones(N) # 初期値
max_loop = 20
error = 1e-15

def newton(F, J, X, x_init, N, error, max_loop):
    # F: 解を求めるための関数
    # J: jacobian行列
    # X: 変数のリスト
    # x_init: 各変数の初期値
    # error: 収束判定の基準
    # max_loop: max_loop
    def variable_num(X, x_init, N): # 変数と初期値の関係を辞書にしておく
        variable_num = {X[i]: x_init[i] for i in range(N)} 
        
        return variable_num
    i = 0
    while np.sum(abs(F_param.subs(variable_num(X, x_init, N)))) >= error:
        
        i += 1
        if i >= max_loop:
            print("Error: Newton nethod did NOT finish normally")
            break
            
        # F_param と J_param　に初期値を代入
        F_init = F.subs(variable_num(X, x_init, N))
        J_init = J.subs(variable_num(X, x_init, N))
        
        # numpyで計算してもらうてもらうために変換
        F_init = np.array(F_init).astype(float)
        J_init = np.array(J_init).astype(float)
        
        # 解く
        x_sol = x_init.reshape(N,1) - np.linalg.solve(J_init, F_init)
        
        x_init = x_sol # 新たな初期値を定義する
            
    return x_sol
    
ss = newton(F_param, J_param, X, x_init, N, error, max_loop)

variable = ["C_star" , "K_star"]
for i in range(N):
    print(variable[i]+ "=" + str(ss[i][0]))

C_star=0.7565354289148452
K_star=1.226144733357211
